# Day 2: Decision Tree from Scratch

**Goal:** Implement a classification decision tree using only NumPy, understanding Gini impurity and recursive splitting before touching sklearn.

In [41]:
import numpy as np 
import pandas as pd 

## Gini Impurity

A decision tree splits data by asking questions. To know which question is best, we need to measure how **mixed** (impure) a group is after the split.

Gini impurity measures exactly that. For a node with $C$ classes:

$$\text{Gini} = 1 - \sum_{c=1}^{C} p_c^2$$

Where $p_c$ is the proportion of samples belonging to class $c$.

- **Gini = 0** → the node is perfectly pure (all one class)
- **Gini = 0.5** → maximum mess for binary classification (50/50 split)

For binary classification (survived / died):

$$\text{Gini} = 1 - p_{\text{survived}}^2 - p_{\text{died}}^2$$

In [42]:
def gini(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0.0

    true = np.sum(y == 1) / len(y)  # get probability for true scores
    false = np.sum(y == 0) / len(y) # same for false

    return 1.0 - true ** 2 - false ** 2

## Weighted Gini Impurity

A single split produces two groups — left and right. We cannot just average their Gini scores equally, because a large impure group is worse than a small impure group.

We weight each group's Gini by how many samples it contains:

$$\text{Weighted Gini} = \frac{n_{\text{left}}}{n} \cdot \text{Gini}_{\text{left}} + \frac{n_{\text{right}}}{n} \cdot \text{Gini}_{\text{right}}$$

The question (feature + threshold) with the **lowest weighted Gini** wins and becomes the split.

In [43]:
def weighted_gini(y_left: np.ndarray, y_right: np.ndarray) -> float:
    n_left = len(y_left)
    n_right = len(y_right)
    n = n_left + n_right

    return (n_left / n) * gini(y_left) + (n_right / n) * gini(y_right)

## Finding the Best Split

For every feature we try all possible thresholds and pick the one that produces the lowest weighted Gini.

- **Categorical features** (dtype = object): try each unique value as threshold — is value == X?
- **Continuous features** (dtype = float/int): try midpoints between every pair of sorted unique values

The winning (feature, threshold) pair becomes the split for the current node.

We also pass `used_features` — a set of categorical features already used on the path from root to this node. Since asking "is female?" inside the Female group tells us nothing, we skip already-used categorical features. Continuous features are never skipped since different thresholds still provide new information.

In [44]:
def best_split(X: pd.DataFrame, y: np.ndarray, feature_names: list[str], used_features: set[str]):
    # we want to find a better impurity than parent node has
    # there is no point of splitting data using the worse impurity 
    best_impurity = gini(y)
    best_feature = None
    best_threshold = None

    for feature in feature_names:
        if not pd.api.types.is_numeric_dtype(X[feature]):
            # if feature is not numeric and was already used to split
            # there is not point of using it since the data is already splitted based on this feature 
            if feature in used_features:
                continue
            else:
                # get unique classes 
                thresholds = X[feature].unique()

                for threshold in thresholds:
                    # create a mask - whether is equal to current class
                    mask = X[feature] == threshold

                    true = y[mask]      # for left node
                    false = y[~mask]    # for right node

                    # data was not splitted so this threshold is useless
                    if len(true) == 0 or len(false) == 0:
                        continue

                    curr_gini = weighted_gini(true, false)
                    if curr_gini < best_impurity:
                        best_impurity = curr_gini
                        best_feature = feature
                        best_threshold = threshold
        else:
            # we need to sort the nums in feature to get midpoints
            sorted_thresholds = np.sort(X[feature].unique())

            # [1, 5, 7, 14] -> [4, 2, 7]
            diffs = np.diff(sorted_thresholds)
            diffs = diffs / 2

            # a + b / 2 = a + (b - a)/ 2
            sorted_thresholds = sorted_thresholds[:-1] + diffs

            # check every midpoint and find the one with the minimal impurity 
            for threshold in sorted_thresholds:
                # in the left node values <= midpoint, right > midpoint
                mask = X[feature] <= threshold

                true = y[mask]
                false = y[~mask]

                # data was not splitted so this threshold is useless
                if len(true) == 0 or len(false) == 0:
                    continue

                curr_gini = weighted_gini(true, false)
                if curr_gini < best_impurity:
                    best_impurity = curr_gini
                    best_feature = feature
                    best_threshold = threshold

    return best_feature, best_threshold

## Building the Tree Recursively

The tree is built recursively. At each node we:

1. Check stopping conditions — if the node is pure, too small, or we hit `max_depth`, we stop and store the majority class as the prediction (a **leaf node**)
2. Otherwise find the best split
3. Divide the data into left and right groups
4. Recurse independently on left and right

Each node is a dictionary — either a leaf `{"leaf": True, "prediction": 0/1}` or an internal node `{"leaf": False, "feature": ..., "threshold": ..., "left": ..., "right": ...}`.

In [ ]:
def build_tree(X: pd.DataFrame, y: np.ndarray, feature_names: list[str], used_features: set[str], max_depth: int, depth: int = 0) -> dict:
    # base stopping conditions (Depth limit or min samples limit)
    if depth == max_depth or len(y) < 10:
        prediction = 1 if np.sum(y == 1) >= (len(y) / 2) else 0
        return {"leaf": True, "prediction": prediction}

    # find best split
    feature, threshold = best_split(X, y, feature_names, used_features)

    # if no split was found (data is pure or no better split exists)
    if feature is None:
        prediction = 1 if np.sum(y == 1) >= (len(y) / 2) else 0
        return {"leaf": True, "prediction": prediction}

    # create mask based on feature type
    is_categorical = not pd.api.types.is_numeric_dtype(X[feature])    
    if is_categorical:
        mask = X[feature] == threshold
        # pass a copy to avoid mutating state across parallel branches
        # so both left and right can use same categorical class 
        used_features = used_features.copy()
        used_features.add(feature)
    else:
        mask = X[feature] <= threshold

    left_data, right_data = X[mask], X[~mask]
    true, false = y[mask], y[~mask]

    # guard against empty splits
    if len(left_data) == 0 or len(right_data) == 0:
        prediction = 1 if np.sum(y == 1) >= (len(y) / 2) else 0
        return {"leaf": True, "prediction": prediction}

    # recurse down left and right subtrees
    left = build_tree(left_data, true, feature_names, used_features, max_depth, depth + 1)
    right = build_tree(right_data, false, feature_names, used_features, max_depth, depth + 1)

    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "left": left,
        "right": right
    }

## Prediction

To predict, we walk a single sample down the tree. At each internal node we check the feature value and go left or right until we hit a leaf, which gives us the class prediction.

In [46]:
def predict_sample(sample: pd.Series, node: dict) -> int:
    # if we hit the leaf - return it's prediction
    if node["leaf"]:
        return node["prediction"]

    # otherwise get feature and threshold for this node
    feature, threshold = node["feature"], node["threshold"]

    # if the threshold is str - it is categorical
    if isinstance(threshold, str):
        if sample[feature] == threshold:
            return predict_sample(sample, node=node["left"])
        else:
            return predict_sample(sample, node=node["right"])
    # otherwise it is numeric
    else:
        if sample[feature] <= threshold:
            return predict_sample(sample, node=node["left"])
        else:
            return predict_sample(sample, node=node["right"])


def predict(X: pd.DataFrame, tree: dict) -> list[int]:
    predictions = list()

    # predict the output for each row 
    for _, row in X.iterrows():
        prediction = predict_sample(row, node=tree)
        predictions.append(prediction)

    return predictions

## Testing on a Toy Dataset

To verify that the recursive splitting, tree construction, and prediction logic work correctly, we will test the model on a small synthetic dataset containing both continuous (`Age`) and categorical (`Sex`) features. 

We will train the decision tree, generate predictions, and calculate the overall classification accuracy.

In [47]:
# ceate a dummy dataset
data = {
    'Age': [22, 25, 47, 35, 14, 50, 28, 19, 60, 38],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1, 1, 0, 1, 0, 1]
}
df_dummy = pd.DataFrame(data)

X_dummy = df_dummy[['Age', 'Sex']]
y_dummy = df_dummy['Survived'].to_numpy()
features = ['Age', 'Sex']

# build the tree
tree = build_tree(X_dummy, y_dummy, features, set(), max_depth=3, depth=0)

# predict and evaluate
predictions = predict(X_dummy, tree)

print("Actual:     ", list(y_dummy))
print("Predictions:", predictions)
print("Accuracy:   ", np.mean(predictions == y_dummy))

Actual:      [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
Predictions: [0, 1, 1, 0, 0, 1, 0, 1, 0, 1]
Accuracy:    0.9
